# Predictia categoriei produsului pe baza titlului

**Curs:** Introduction to Machine Learning Using Python
**Modul:** Crearea unui model ML pentru clasificare

Acest notebook documenteaza intregul proces de dezvoltare a unui model care
prezice categoria unui produs (`Category Label`) pe baza titlului sau
(`Product Title`), folosind setul de date `products.csv`.

Structura notebook-ului:
1. Incarcare si explorare date
2. Curatare si pregatire date
3. Inginerie caracteristici (feature engineering)
4. Impartire train/test
5. Vectorizare text + preprocesare
6. Antrenare si comparare modele
7. Evaluare (acuratete, raport de clasificare, matrice de confuzie)
8. Salvarea modelului final
9. Concluzie


## 1. Import biblioteci

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import MinMaxScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score

from features import build_features, NUMERIC_FEATURE_COLUMNS


## 2. Incarcarea si explorarea datelor

Incarcam `products.csv` si aruncam o privire generala: cate produse avem,
cate categorii distincte, si daca exista valori lipsa in coloanele care ne
intereseaza (`Product Title`, `Category Label`).

In [ ]:
TITLE_COLUMN = "Product Title"
CATEGORY_COLUMN = "Category Label"

df = pd.read_csv("products.csv")

# IMPORTANT: products.csv are spatii ascunse in unele nume de coloane
# (ex: " Category Label" in loc de "Category Label"). Le curatam imediat.
df.columns = df.columns.str.strip()

print(f"Numar total de produse: {len(df)}")
display(df.head())

print("\nValori lipsa pe coloane:")
print(df.isnull().sum())

print(f"\nNumar de categorii distincte: {df[CATEGORY_COLUMN].nunique()}")


### Distributia categoriilor

Cate produse are fiecare categorie? E important sa stim daca unele categorii au foarte putine exemple - acestea sunt mai greu de invatat de catre model.

In [ ]:
category_counts = df[CATEGORY_COLUMN].value_counts()
print(category_counts)

plt.figure(figsize=(10, 6))
category_counts.head(20).plot(kind="barh")
plt.title("Top 20 categorii dupa numarul de produse")
plt.xlabel("Numar de produse")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()


## 3. Curatarea datelor

- Eliminam produsele fara titlu sau fara categorie.
- Standardizam textul categoriei (eliminam spatii in plus).
- Eliminam categoriile cu mai putin de 2 produse - acestea nu pot fi
  impartite corect intre setul de antrenament si cel de testare
  (train_test_split cu stratify are nevoie de cel putin 2 exemple per clasa).

In [ ]:
df = df.dropna(subset=[TITLE_COLUMN, CATEGORY_COLUMN])
df[CATEGORY_COLUMN] = df[CATEGORY_COLUMN].astype(str).str.strip()

# PROBLEMA GASITA IN DATELE REALE: aceeasi categorie apare scrisa in mai
# multe feluri - ex: "CPU" vs "CPUs", "Mobile Phone" vs "Mobile Phones",
# "fridge" (litere mici) vs "Fridges". Le normalizam si le unificam sub
# forma cea mai frecventa, ca sa nu creeze categorii artificiale separate.
def _normalize_category(cat):
    cat = cat.strip().lower()
    if cat.endswith("s") and not cat.endswith("ss"):
        cat = cat[:-1]
    return cat

df["_category_normalized"] = df[CATEGORY_COLUMN].apply(_normalize_category)
canonical_names = (
    df.groupby("_category_normalized")[CATEGORY_COLUMN]
    .agg(lambda values: values.value_counts().idxmax())
)
df[CATEGORY_COLUMN] = df["_category_normalized"].map(canonical_names)
df = df.drop(columns=["_category_normalized"])

valid_categories = category_counts[category_counts >= 2].index
df = df[df[CATEGORY_COLUMN].isin(valid_categories)]

print(f"Produse ramase dupa curatare: {len(df)}")
print(f"Categorii ramase: {df[CATEGORY_COLUMN].nunique()}")


## 4. Ingineria caracteristicilor (Feature Engineering)

Pe langa textul titlului, adaugam caracteristici suplimentare care ar putea
ajuta modelul sa distinga mai bine intre categorii:

- **title_word_count** - numarul de cuvinte din titlu (titluri tehnice, cu
  multe specificatii, pot avea mai multe cuvinte)
- **title_char_count** - lungimea totala a titlului
- **has_digit** - daca titlul contine cifre (frecvent la modele/specificatii,
  ex: "128GB", "A52")
- **has_special_char** - daca titlul contine caractere speciale
- **has_all_caps_word** - daca exista un cuvant scris integral cu majuscule
  (ex: "USB", "LED", "HD") - poate indica anumite categorii de electronice
- **longest_word_length** - lungimea celui mai lung cuvant

Functia `build_features` e definita in `features.py`, ca sa fie reutilizata
identic si de scripturile `train_model.py` / `predict_category.py`.

In [ ]:
df = build_features(df, title_column=TITLE_COLUMN)
display(df[[TITLE_COLUMN] + NUMERIC_FEATURE_COLUMNS].head())


## 5. Impartirea datelor (train/test split)

Impartim in 80% antrenament / 20% testare, cu `stratify` pe categorie -
astfel, fiecare categorie e reprezentata proportional in ambele seturi.

In [ ]:
feature_columns = [TITLE_COLUMN] + NUMERIC_FEATURE_COLUMNS
X = df[feature_columns]
y = df[CATEGORY_COLUMN]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Produse de antrenament: {len(X_train)}")
print(f"Produse de testare: {len(X_test)}")


## 6. Preprocesare - TF-IDF + caracteristici numerice

Folosim `ColumnTransformer` ca sa aplicam:
- `TfidfVectorizer` pe coloana de text (`Product Title`)
- `StandardScaler` pe caracteristicile numerice

Rezultatul e combinat automat intr-o singura matrice de caracteristici,
gata de folosit de orice model de clasificare.

In [ ]:
# Folosim MinMaxScaler (nu StandardScaler) pentru caracteristicile numerice,
# deliberat: StandardScaler poate produce valori negative, iar unul din
# modelele comparate mai jos (MultinomialNB) accepta DOAR valori nenegative.
preprocessor = ColumnTransformer(transformers=[
    ("tfidf", TfidfVectorizer(max_features=5000), TITLE_COLUMN),
    ("numeric", MinMaxScaler(), NUMERIC_FEATURE_COLUMNS),
])


## 7. Antrenarea si compararea mai multor modele

Testam trei algoritmi diferiti, fiecare impachetat intr-un `Pipeline`
(preprocesare + model), ca sa putem compara corect performanta lor:

- **Logistic Regression** - standard, rapid, robust
- **Multinomial Naive Bayes** - clasic pentru clasificare de text
- **Linear SVC** - de multe ori foarte performant pe text vectorizat TF-IDF

Evaluam fiecare model pe: acuratete, F1 (weighted - relevant cand avem multe
categorii, posibil dezechilibrate) si raport de clasificare complet.

In [ ]:
candidate_models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Multinomial Naive Bayes": MultinomialNB(),
    "Linear SVC": LinearSVC(),
}

trained_pipelines = {}
results_summary = []

for name, classifier in candidate_models.items():
    pipeline = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("classifier", classifier),
    ])
    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)

    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average="weighted")

    trained_pipelines[name] = (pipeline, y_pred)
    results_summary.append({"Model": name, "Acuratete": acc, "F1 (weighted)": f1})

    print("=" * 70)
    print(f"Model: {name}")
    print(f"Acuratete: {acc:.4f}  |  F1 (weighted): {f1:.4f}")
    print("\nRaport de clasificare (rezumat):")
    print(classification_report(y_test, y_pred, zero_division=0))


### Tabel comparativ

In [ ]:
summary_df = pd.DataFrame(results_summary).sort_values(by="F1 (weighted)", ascending=False)
display(summary_df)


## 8. Matricea de confuzie pentru cel mai bun model

Cu multe categorii, o matrice de confuzie completa poate fi greu de citit -
ne concentram pe **top 15 cele mai frecvente categorii**, ca sa pastram
graficul lizibil.

In [ ]:
best_model_name = summary_df.iloc[0]["Model"]
best_pipeline, best_y_pred = trained_pipelines[best_model_name]

print(f"Cel mai bun model: {best_model_name}")

top_categories = y_test.value_counts().head(15).index.tolist()

cm = confusion_matrix(y_test, best_y_pred, labels=top_categories)

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=top_categories, yticklabels=top_categories)
plt.title(f"Matrice de confuzie - {best_model_name} (top 15 categorii)")
plt.xlabel("Predictie")
plt.ylabel("Valoare reala")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()


## 9. Salvarea modelului final

Salvam pipeline-ul castigator (preprocesare + model antrenat) intr-un fisier
`.pkl`, ca sa poata fi incarcat direct de `predict_category.py`, fara sa
retrenam nimic.

In [ ]:
import pickle

with open("category_model.pkl", "wb") as f:
    pickle.dump(best_pipeline, f)

print("Model salvat in category_model.pkl")


## 10. Concluzie

*(Completeaza aceasta sectiune cu rezultatele reale obtinute la rularea ta -
numerele pot varia usor fata de exemplele generice de mai jos.)*

- **Modelul cu cea mai buna performanta a fost:** ______ (numele modelului,
  acuratetea si F1 din tabelul comparativ de mai sus).
- **Cum s-au comportat celelalte modele:** ______
- **Ce categorii sunt cel mai des confundate intre ele** (din matricea de
  confuzie): ______
- **Ce caracteristici suplimentare au avut impact** (sau nu) asupra
  performantei: ______
- **Modelul final ales pentru echipa:** ______, pentru ca ______

### Observatii si dileme intalnite pe parcurs
- Cea mai mare provocare a fost: ______
- Ce as imbunatati intr-o versiune viitoare: ______ (ex: mai multe
  caracteristici, un alt algoritm precum Random Forest, tuning de
  hiperparametri, folosirea si a altor coloane precum Merchant Rating)
